# CLM — Correlation Learning Mechanism (CNN + ANN)


## 0. Mount Google Drive & Set Dataset Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

OUTPUT_DIR = '/content/drive/MyDrive/Tugas Akhir/CLM/FINAL_CLM_test'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# folder dataset augmented untuk training
TRAIN_DIR_AUG = os.path.join(OUTPUT_DIR, 'dataset_augmented', 'train')

# TARGET_MULTIPLIER = 4 -> total = 1 asli + 3 augmented
TARGET_MULTIPLIER = 4
AUG_PER_METHOD    = TARGET_MULTIPLIER - 1
AUG_SEED          = 42

# grid search 3 hyperparameter utama CLM
PARAM_GRID_CLM = {
    'adam_lr'    : [0.0001, 0.00025, 0.001],
    'dropout_p'  : [0.05, 0.10, 0.20],
    'total_iter' : [3000, 5000, 10000],
}

# True -> 7 kombinasi penting saja, False -> semua 27 kombinasi
REDUCED_GRID = True

# subset REDUCED_GRID: baseline paper + variasi 1 param sekaligus
REDUCED_COMBINATIONS_CLM = [
    (0.00025,  0.10,      5000),  # baseline
    (0.0001,   0.10,      5000),
    (0.001,    0.10,      5000),
    (0.00025,  0.05,      5000),
    (0.00025,  0.20,      5000),
    (0.00025,  0.10,      3000),
    (0.00025,  0.10,      10000),
]

val_no_tumor  = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_test/val/no_tumor'
val_tumor     = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_test/val/tumor'

train_no_tumor = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_test/train/no_tumor'
train_tumor    = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_test/train/tumor'

test_no_tumor  = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_test/test/no_tumor'
test_tumor     = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_test/test/tumor'

TRAIN_DIR = os.path.dirname(train_no_tumor)
VAL_DIR   = os.path.dirname(val_no_tumor)
TEST_DIR  = os.path.dirname(test_no_tumor)

print(f'Output: {OUTPUT_DIR} | REDUCED_GRID={REDUCED_GRID}')


In [ ]:
import subprocess, sys

try:
    import cupy as cp
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda11x'])
    import cupy as cp

import torch
USE_GPU = torch.cuda.is_available()
if not USE_GPU:
    print('Peringatan: GPU tidak tersedia, fallback ke CPU')

# xp = cupy jika GPU tersedia, numpy jika tidak
import numpy as np
xp = cp if USE_GPU else np
print(f'Backend aktif: {"CuPy (GPU)" if USE_GPU else "NumPy (CPU)"}')


## 1. Imports & Logging Setup

In [ ]:
import copy
import glob
import logging
import os
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F  # batched conv2d/pool2d

import cupy as cp
import cupyx.scipy.signal as cusignal   # konvolusi GPU
from scipy.signal import convolve2d as cpu_convolve2d  # fallback CPU

from tqdm.auto import tqdm
import cv2
import random
import shutil
import itertools
import json as _json
import pandas as pd

# level WARNING agar log progres tidak membanjiri output
logging.basicConfig(
    level=logging.WARNING,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)


## 2. Filter Definitions

In [ ]:
FILTERS: Dict[int, Tuple[str, np.ndarray]] = {
    # filter 3x3
    1:  ("Vertical",           np.array([[-1,  0,  1],
                                          [-1,  0,  1],
                                          [-1,  0,  1]], dtype=np.float32)),
    2:  ("Left Diagonal",      np.array([[-1, -1,  0],
                                          [-1,  0,  1],
                                          [ 0,  1,  1]], dtype=np.float32)),
    3:  ("Right Diagonal",     np.array([[ 0, -1, -1],
                                          [ 1,  0, -1],
                                          [ 1,  1,  0]], dtype=np.float32)),
    4:  ("Gradient North",     np.array([[-1, -1, -1],
                                          [ 0,  0,  0],
                                          [ 1,  1,  1]], dtype=np.float32)),
    5:  ("Gradient West",      np.array([[-1,  0,  1],
                                          [-1,  0,  1],
                                          [-1,  0,  1]], dtype=np.float32)),
    6:  ("Gradient East",      np.array([[ 1,  0, -1],
                                          [ 1,  0, -1],
                                          [ 1,  0, -1]], dtype=np.float32)),
    7:  ("Gradient South",     np.array([[ 1,  1,  1],
                                          [ 0,  0,  0],
                                          [-1, -1, -1]], dtype=np.float32)),
    8:  ("Gradient NE",        np.array([[ 0,  1,  1],
                                          [-1,  0,  1],
                                          [-1, -1,  0]], dtype=np.float32)),
    9:  ("Gradient SW",        np.array([[ 0, -1, -1],
                                          [ 1,  0, -1],
                                          [ 1,  1,  0]], dtype=np.float32)),
    10: ("Smooth 3×3",         np.ones((3, 3), dtype=np.float32) / 9.0),
    11: ("High-Pass 3×3",      np.array([[-1, -1, -1],
                                          [-1,  8, -1],
                                          [-1, -1, -1]], dtype=np.float32)),
    12: ("Laplacian 3×3",      np.array([[ 0,  1,  0],
                                          [ 1, -4,  1],
                                          [ 0,  1,  0]], dtype=np.float32)),
    13: ("Lower 3×3",          np.array([[ 0,  0,  0],
                                          [ 0,  1,  0],
                                          [ 1,  1,  1]], dtype=np.float32) / 4.0),
    14: ("Upper 3×3",          np.array([[ 1,  1,  1],
                                          [ 0,  1,  0],
                                          [ 0,  0,  0]], dtype=np.float32) / 4.0),
    # filter 5x5
    50: ("Smooth 5×5",         np.ones((5, 5), dtype=np.float32) / 25.0),
    51: ("High-Pass 5×5",      np.array([[-1, -1, -1, -1, -1],
                                          [-1, -1, -1, -1, -1],
                                          [-1, -1, 24, -1, -1],
                                          [-1, -1, -1, -1, -1],
                                          [-1, -1, -1, -1, -1]], dtype=np.float32)),
    52: ("Laplacian 5×5",      np.array([[ 0,  0,  1,  0,  0],
                                          [ 0,  1,  2,  1,  0],
                                          [ 1,  2,-16,  2,  1],
                                          [ 0,  1,  2,  1,  0],
                                          [ 0,  0,  1,  0,  0]], dtype=np.float32)),
}

VALID_FILTER_IDS: List[int] = sorted(FILTERS.keys())

# upload kernel filter ke GPU (jika tersedia) agar konvolusi lebih cepat
if USE_GPU:
    GPU_FILTERS = {fid: (name, cp.asarray(kernel)) for fid, (name, kernel) in FILTERS.items()}
else:
    GPU_FILTERS = FILTERS


## 3. Palette

In [ ]:
class PaletteLayer:
    """Satu layer palette CNN (pool | conv | relu)."""
    def __init__(self, kind: str, params):
        self.kind   = kind
        self.params = params


class Palette:
    """Palette CNN — urutan operasi yang diterapkan ke tiap segmen citra."""
    def __init__(self, layers: Optional[List[PaletteLayer]] = None):
        self.layers: List[PaletteLayer] = layers if layers is not None else self._best_palette()

    @staticmethod
    def _best_palette() -> List[PaletteLayer]:
        # palette terbaik dari paper (§4.2 / Fig. 2)
        return [
            PaletteLayer("pool", (4, 4)),
            PaletteLayer("conv", [1, 11, 6, 3]),
            PaletteLayer("relu", None),
            PaletteLayer("pool", (3, 3)),
            PaletteLayer("conv", [52, 0, 1, 8]),   # id=0 -> identity (pass-through)
        ]

    def encode(self) -> str:
        parts = []
        for lyr in self.layers:
            if lyr.kind == "pool":
                parts.append(f"[{lyr.params[0]};{lyr.params[1]}]")
            elif lyr.kind == "conv":
                parts.append("c[" + ";".join(str(i) for i in lyr.params) + "]")
            elif lyr.kind == "relu":
                parts.append("r")
        return ", ".join(parts)

    @staticmethod
    def random(n_conv: int = 2, filters_per_conv: int = 4) -> "Palette":
        layers: List[PaletteLayer] = []
        pool_choices = [3, 4]
        for i in range(n_conv):
            k = int(np.random.choice(pool_choices))
            layers.append(PaletteLayer("pool", (k, k)))
            fids = [int(np.random.choice(VALID_FILTER_IDS))
                    for _ in range(filters_per_conv)]
            layers.append(PaletteLayer("conv", fids))
            if i < n_conv - 1:
                layers.append(PaletteLayer("relu", None))
        return Palette(layers)


## 4. CNN Component

In [ ]:
class CNN:
    """Komponen CNN dari CLM — dipercepat via CuPy/Torch.
    forward_batch() memproses semua gambar sekaligus (batched conv2d/pool2d),
    menggantikan forward() yang looping per-gambar (lebih lambat, dipakai fallback
    untuk predict/mark_detection single-image).
    """
    NUM_SEGMENTS: int = 64
    GRID: int = 8

    def __init__(self):
        self._device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # pre-konversi kernel filter ke torch tensor di GPU (1,1,kH,kW)
        self._torch_filters: Dict[int, torch.Tensor] = {}
        for fid, (_, kernel) in FILTERS.items():
            t = torch.from_numpy(kernel.copy()).float().to(self._device)
            self._torch_filters[fid] = t.unsqueeze(0).unsqueeze(0)

    # metode per-gambar (dipakai predict/mark_detection)
    def _apply_filter(self, img: cp.ndarray, fid: int) -> cp.ndarray:
        if fid == 0 or fid not in GPU_FILTERS:
            return img.astype(cp.float32)
        _, kernel = GPU_FILTERS[fid]
        return cusignal.convolve2d(img.astype(cp.float32), kernel,
                                   mode='same', boundary='symm')

    def _max_pool(self, img: cp.ndarray, k: int) -> cp.ndarray:
        h, w = img.shape
        nh, nw = h // k, w // k
        if nh == 0 or nw == 0:
            return img
        return img[:nh*k, :nw*k].reshape(nh, k, nw, k).max(axis=(1, 3))

    @staticmethod
    def _relu(img: cp.ndarray) -> cp.ndarray:
        return cp.maximum(0.0, img)

    def _conv_layer(self, maps: List[cp.ndarray], fids: List[int]) -> List[cp.ndarray]:
        out: List[cp.ndarray] = []
        for fm in maps:
            for fid in fids:
                out.append(self._apply_filter(fm, fid))
        return out

    def _statistics(self, maps: List[cp.ndarray]) -> cp.ndarray:
        # 6 statistik per feature map: min, max, sum, median, mean, std
        # ambil elemen nonzero dulu, fallback ke flat hanya jika semua piksel nol
        stats: List[float] = []
        for fm in maps:
            flat = fm.flatten()
            nz   = flat[flat != 0.0]
            nz   = nz if nz.size > 0 else flat
            stats += [
                float(cp.min(nz)), float(cp.max(nz)), float(cp.sum(nz)),
                float(cp.median(nz)), float(cp.mean(nz)), float(cp.std(nz)),
            ]
        return cp.array(stats, dtype=cp.float32)

    def segment(self, image: np.ndarray) -> List[cp.ndarray]:
        # segmentasi gambar -> 64 fragmen, upload ke GPU
        if image.ndim == 3:
            image = np.mean(image, axis=2)
        h, w = image.shape
        side = min(h, w)
        image = image[:side, :side]
        seg_sz = side // self.GRID
        segs: List[cp.ndarray] = []
        for r in range(self.GRID):
            for c in range(self.GRID):
                seg = image[r*seg_sz:(r+1)*seg_sz, c*seg_sz:(c+1)*seg_sz].astype(np.float32)
                segs.append(cp.asarray(seg))
        return segs

    def forward(self, image: np.ndarray, palette: 'Palette') -> np.ndarray:
        segments = self.segment(image)
        all_stats: List[cp.ndarray] = []
        for seg in segments:
            current: List[cp.ndarray] = [seg]
            for lyr in palette.layers:
                if lyr.kind == 'pool':
                    k = lyr.params[0]
                    current = [self._max_pool(m, k) for m in current]
                elif lyr.kind == 'conv':
                    current = self._conv_layer(current, lyr.params)
                elif lyr.kind == 'relu':
                    current = [self._relu(m) for m in current]
            all_stats.append(self._statistics(current))
        vec = cp.concatenate(all_stats)
        mx = cp.max(cp.abs(vec))
        if mx > 0:
            vec /= mx
        return cp.asnumpy(vec)

    # metode batched (dipakai saat training, jauh lebih cepat)
    def pre_segment_images(self, images: List[np.ndarray]) -> torch.Tensor:
        """Segmentasi semua gambar sekaligus -> (N x 64, 1, H_seg, W_seg) di GPU,
        dipanggil sekali di awal training lalu hasilnya di-cache."""
        # validasi semua gambar berukuran sama; jika tidak, seg_sz berbeda
        # dan fitur geometri segmen akan terdistorsi secara diam-diam
        sides: List[int] = []
        for img in images:
            h = img.shape[0]
            w = img.shape[1]
            sides.append(min(h, w))
        unique_sides = set(sides)
        if len(unique_sides) > 1:
            raise ValueError(
                f"pre_segment_images: semua gambar harus berukuran sama. "
                f"Ditemukan sisi efektif berbeda: {sorted(unique_sides)}. "
                f"Lakukan resize seragam sebelum memanggil fit() / pre_segment_images()."
            )

        all_segs: List[np.ndarray] = []
        target_sz: Optional[int] = None
        for img in images:
            if img.ndim == 3:
                img = np.mean(img, axis=2)
            h, w = img.shape
            side = min(h, w)
            img = img[:side, :side]
            seg_sz = side // self.GRID
            if target_sz is None:
                target_sz = seg_sz
            for r in range(self.GRID):
                for c in range(self.GRID):
                    seg = img[r*seg_sz:(r+1)*seg_sz, c*seg_sz:(c+1)*seg_sz].astype(np.float32)
                    all_segs.append(seg)
        stacked = np.stack(all_segs, axis=0)          # (N*64, H, W)
        return torch.from_numpy(stacked).unsqueeze(1).to(self._device)

    def _batch_conv_layer(self, maps: torch.Tensor, fids: List[int]) -> torch.Tensor:
        # grouped conv2d batched: (B,C,H,W) -> (B, C*len(fids), H, W)
        C = maps.shape[1]
        out_parts: List[torch.Tensor] = []
        for fid in fids:
            if fid == 0 or fid not in self._torch_filters:
                out_parts.append(maps)
            else:
                k = self._torch_filters[fid]
                pad = k.shape[-1] // 2
                kg = k.expand(C, 1, k.shape[2], k.shape[3])
                out_parts.append(F.conv2d(maps, kg, padding=pad, groups=C))
        return torch.cat(out_parts, dim=1)

    @staticmethod
    def _batch_stats(maps: torch.Tensor) -> torch.Tensor:
        # 6 statistik per feature map untuk semua sample sekaligus: (B,C,H,W) -> (B,C*6)
        B, C, H, W = maps.shape
        flat = maps.view(B, C, -1)
        mins    = flat.min(dim=-1).values
        maxs    = flat.max(dim=-1).values
        sums    = flat.sum(dim=-1)
        means   = flat.mean(dim=-1)
        stds    = flat.std(dim=-1, unbiased=False)

        # median hanya dari elemen nonzero (konsisten dengan _statistics()):
        # sort ascending -> nol berkumpul di depan, cari median dari sisi nonzero
        sorted_flat, _ = flat.sort(dim=-1)
        n_nonzero = (flat != 0.0).sum(dim=-1).clamp(min=1)
        hw = flat.shape[-1]
        start_idx = hw - n_nonzero
        mid_offset = n_nonzero // 2
        median_idx = (start_idx + mid_offset).clamp(0, hw - 1)
        medians = sorted_flat.gather(-1, median_idx.unsqueeze(-1)).squeeze(-1)
        # jumlah nonzero genap -> rata-rata dua elemen tengah
        mid_offset2 = ((n_nonzero - 1) // 2).clamp(min=0)
        median_idx2 = (start_idx + mid_offset2).clamp(0, hw - 1)
        medians2    = sorted_flat.gather(-1, median_idx2.unsqueeze(-1)).squeeze(-1)
        even_mask = (n_nonzero % 2 == 0)
        medians   = torch.where(even_mask, (medians + medians2) * 0.5, medians)

        return torch.stack([mins, maxs, sums, medians, means, stds], dim=2).view(B, C * 6)

    def forward_batch(self, n_images: int, palette: 'Palette',
                      seg_cache: torch.Tensor) -> np.ndarray:
        """Forward pass untuk semua gambar sekaligus (jauh lebih cepat dari forward())."""
        maps = seg_cache   # (N*64, 1, H_seg, W_seg), sudah di GPU

        for lyr in palette.layers:
            if lyr.kind == 'pool':
                k = lyr.params[0]
                if maps.shape[2] // k > 0 and maps.shape[3] // k > 0:
                    maps = F.max_pool2d(maps, kernel_size=k, stride=k)
            elif lyr.kind == 'conv':
                maps = self._batch_conv_layer(maps, lyr.params)
            elif lyr.kind == 'relu':
                maps = F.relu(maps)

        stats = self._batch_stats(maps)                         # (N*64, C*6)
        features = stats.view(n_images, -1)                     # (N, 64*C*6)

        # normalisasi per-sample
        mx = features.abs().max(dim=1, keepdim=True).values.clamp(min=1e-12)
        features = features / mx

        return features.cpu().numpy().astype(np.float32)


## 5. ANN Component

In [ ]:
class ANN:
    """ANN — komponen klasifikasi CLM, GPU-accelerated (CuPy).
    Arsitektur: [90][45][10][5][4][2], aktivasi sigmoid, loss cross-entropy, training ADAM.
    """
    DEFAULT_LAYERS: List[int] = [90, 45, 10, 5, 4, 2]

    def __init__(self, input_size: int, layer_sizes: Optional[List[int]] = None,
                 dropout_p: float = 0.10):
        self.input_size  = input_size
        self.layer_sizes = layer_sizes or self.DEFAULT_LAYERS
        self.dropout_p   = dropout_p
        dims = [input_size] + self.layer_sizes
        self.W: List[cp.ndarray] = []
        self.b: List[cp.ndarray] = []
        for i in range(len(dims) - 1):
            scale = np.sqrt(2.0 / (dims[i] + dims[i+1]))
            self.W.append(cp.array(
                np.random.randn(dims[i], dims[i+1]).astype(np.float32) * scale))
            self.b.append(cp.zeros(dims[i+1], dtype=cp.float32))
        self._adam_reset()

    @staticmethod
    def sigmoid(e: cp.ndarray) -> cp.ndarray:
        return 1.0 / (1.0 + cp.exp(-cp.clip(e, -500, 500)))

    @staticmethod
    def _dsigmoid(sig: cp.ndarray) -> cp.ndarray:
        # sig harus berupa OUTPUT sigmoid, bukan pre-activation
        return sig * (1.0 - sig)

    def forward(self, X: cp.ndarray, training: bool = False
                ) -> Tuple[cp.ndarray, List[cp.ndarray]]:
        # acts[0]=X, acts[i+1]=sigma(acts[i]@W[i]+b[i]) -> acts[i] = input ke layer i
        acts: List[cp.ndarray] = [X]
        cur = X
        for i, (Wi, bi) in enumerate(zip(self.W, self.b)):
            cur = self.sigmoid(cur @ Wi + bi)
            if training and self.dropout_p > 0 and i < len(self.W) - 1:
                # inverted dropout: bagi (1-p) agar expected value output
                # konsisten antara training dan inference
                mask = (cp.random.rand(*cur.shape) > self.dropout_p).astype(cp.float32)
                cur  = cur * mask / (1.0 - self.dropout_p)
            acts.append(cur)
        return cur, acts

    @staticmethod
    def cross_entropy(y_pred: cp.ndarray, y_true: cp.ndarray,
                      class_weights: Optional[cp.ndarray] = None) -> float:
        # binary cross-entropy, class_weights untuk mengatasi class imbalance
        eps = 1e-12
        yp  = cp.clip(y_pred, eps, 1.0 - eps)
        ce  = -y_true * cp.log(yp) - (1.0 - y_true) * cp.log(1.0 - yp)  # (N, C)
        if class_weights is not None:
            sample_w = y_true @ class_weights          # (N,) bobot per sample
            ce = ce * sample_w[:, None]
        return float(cp.mean(ce))

    def adam_step(self, X: cp.ndarray, y: cp.ndarray,
                  lr: float = 0.00025, beta1: float = 0.89,
                  beta2: float = 0.995, eps: float = 1e-7,
                  class_weights: Optional[cp.ndarray] = None) -> float:
        out, acts = self.forward(X, training=True)
        loss  = self.cross_entropy(out, y, class_weights)

        delta = out - y                                # gradient di output layer
        if class_weights is not None:
            sample_w = y @ class_weights
            delta    = delta * sample_w[:, None]        # skala gradient kelas minoritas

        gW: List[cp.ndarray] = [None] * len(self.W)   # type: ignore
        gb: List[cp.ndarray] = [None] * len(self.b)   # type: ignore

        for i in reversed(range(len(self.W))):
            gW[i] = acts[i].T @ delta / X.shape[0]
            gb[i] = cp.mean(delta, axis=0)
            # delta dipropagasi penuh sampai layer 0 (tanpa skip i=0), acts[i]
            # adalah input ke layer i sehingga dsigmoid(acts[i]) benar secara dimensi
            delta = (delta @ self.W[i].T) * self._dsigmoid(acts[i])

        self._t += 1
        t = self._t
        for i in range(len(self.W)):
            self._m_W[i] = beta1 * self._m_W[i] + (1 - beta1) * gW[i]
            self._v_W[i] = beta2 * self._v_W[i] + (1 - beta2) * gW[i] ** 2
            self._m_b[i] = beta1 * self._m_b[i] + (1 - beta1) * gb[i]
            self._v_b[i] = beta2 * self._v_b[i] + (1 - beta2) * gb[i] ** 2
            mh_W = self._m_W[i] / (1 - beta1 ** t)
            vh_W = self._v_W[i] / (1 - beta2 ** t)
            mh_b = self._m_b[i] / (1 - beta1 ** t)
            vh_b = self._v_b[i] / (1 - beta2 ** t)
            self.W[i] -= lr * mh_W / (cp.sqrt(vh_W) + eps)
            self.b[i] -= lr * mh_b / (cp.sqrt(vh_b) + eps)
        return loss

    def _adam_reset(self):
        self._m_W = [cp.zeros_like(w) for w in self.W]
        self._v_W = [cp.zeros_like(w) for w in self.W]
        self._m_b = [cp.zeros_like(b) for b in self.b]
        self._v_b = [cp.zeros_like(b) for b in self.b]
        self._t   = 0

    def predict(self, X: cp.ndarray) -> cp.ndarray:
        out, _ = self.forward(X, training=False)
        return out

    def clone(self) -> 'ANN':
        new = ANN(self.input_size, self.layer_sizes, self.dropout_p)
        new.W = [w.copy() for w in self.W]
        new.b = [b.copy() for b in self.b]
        return new

    def absorb_mean(self, others: List['ANN']) -> None:
        pool = [self] + others
        for i in range(len(self.W)):
            self.W[i] = cp.mean(cp.stack([m.W[i] for m in pool]), axis=0)
            self.b[i] = cp.mean(cp.stack([m.b[i] for m in pool]), axis=0)


## 6. Palette Archives & Generator

In [ ]:
class PaletteArchives:
    """Menyimpan triple (accuracy, loss, palette) terbaik."""

    def __init__(self, max_size: int = 200):
        self.max_size = max_size
        self._entries: List[Tuple[float, float, Palette]] = []

    def add(self, accuracy: float, loss: float, palette: Palette) -> None:
        self._entries.append((accuracy, loss, palette))
        self._entries.sort(key=lambda e: (-e[0], e[1]))
        self._entries = self._entries[:self.max_size]

    def best_palettes(self, n: int = 10) -> List[Palette]:
        return [e[2] for e in self._entries[:n]]

    def best_accuracy(self) -> float:
        return self._entries[0][0] if self._entries else 0.0

    def is_empty(self) -> bool:
        return len(self._entries) == 0


class PaletteGenerator:
    """Menghasilkan palette dari archive + reshuffling. Makin rendah efisiensi ANN, makin banyak reshuffling."""
    def __init__(self, archives: PaletteArchives):
        self.archives = archives

    def generate(self, n: int, ann_efficiency: float = 1.0) -> List[Palette]:
        if self.archives.is_empty():
            return [Palette.random() for _ in range(n)]
        reshuffle_rate = max(0.0, 1.0 - ann_efficiency)
        seeds = self.archives.best_palettes(min(10, len(self.archives._entries)))
        palettes: List[Palette] = []
        for i in range(n):
            base = seeds[i % len(seeds)]
            if np.random.rand() < reshuffle_rate:
                palettes.append(self._reshuffle(base, reshuffle_rate))
            else:
                palettes.append(copy.deepcopy(base))
        return palettes

    @staticmethod
    def _reshuffle(palette: Palette, intensity: float) -> Palette:
        new_layers: List[PaletteLayer] = []
        for lyr in palette.layers:
            if lyr.kind == "conv" and np.random.rand() < intensity:
                new_fids = [
                    int(np.random.choice(VALID_FILTER_IDS))
                    if np.random.rand() < intensity else fid
                    for fid in lyr.params
                ]
                new_layers.append(PaletteLayer("conv", new_fids))
            elif lyr.kind == "pool" and np.random.rand() < intensity * 0.3:
                k = int(np.random.choice([3, 4]))
                new_layers.append(PaletteLayer("pool", (k, k)))
            else:
                new_layers.append(copy.deepcopy(lyr))
        return Palette(new_layers)


## 7. GPU Batch Training

In [ ]:
class GPUBatchTrainer:
    """Trainer batch GPU-native untuk ANN."""
    def __init__(self, n_threads: Optional[int] = None,
                 iters_per_round: int = 100,
                 lr: float = 0.00025, beta1: float = 0.89,
                 beta2: float = 0.995, eps: float = 1e-7):
        self.n      = 1
        self.iters  = iters_per_round
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps

    def train_round(self, ann: ANN, X_np: np.ndarray, y_np: np.ndarray,
                    class_weights: Optional[cp.ndarray] = None
                    ) -> Tuple[ANN, float, float]:
        X = cp.asarray(X_np, dtype=cp.float32)
        y = cp.asarray(y_np, dtype=cp.float32)

        trained    = ann.clone()
        total_loss = 0.0
        for _ in tqdm(range(self.iters), desc='ADAM steps', leave=False, unit='step'):
            total_loss += trained.adam_step(
                X, y,
                lr=self.lr, beta1=self.beta1,
                beta2=self.beta2, eps=self.eps,
                class_weights=class_weights,
            )

        preds    = cp.argmax(trained.predict(X), axis=1)
        truths   = cp.argmax(y, axis=1)
        avg_acc  = float(cp.mean(preds == truths))
        avg_loss = total_loss / self.iters     # loss dirata-ratakan seluruh iterasi
        return trained, avg_loss, avg_acc


## 8. Metrics

In [ ]:
def compute_metrics(y_true_oh: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Hitung metrik (Table 2). Otomatis konversi CuPy ke NumPy jika perlu."""
    # Konversi ke NumPy jika input adalah CuPy array
    if hasattr(y_true_oh, 'get'): y_true_oh = y_true_oh.get()
    if hasattr(y_pred,    'get'): y_pred    = y_pred.get()

    pred_lbl = np.argmax(y_pred,    axis=1)
    true_lbl = np.argmax(y_true_oh, axis=1)
    TP = np.sum((pred_lbl == 1) & (true_lbl == 1))
    TN = np.sum((pred_lbl == 0) & (true_lbl == 0))
    FP = np.sum((pred_lbl == 1) & (true_lbl == 0))
    FN = np.sum((pred_lbl == 0) & (true_lbl == 1))
    e = 1e-12
    accuracy    = (TP + TN) / (TP + TN + FP + FN + e)
    precision   = TP / (TP + FP + e)
    recall      = TP / (TP + FN + e)
    specificity = TN / (TN + FP + e)
    f1          = 2 * precision * recall / (precision + recall + e)
    fpr         = FP / (FP + TN + e)
    fnr         = FN / (FN + TP + e)
    fdr         = FP / (FP + TP + e)
    npv         = TN / (TN + FN + e)
    for_        = FN / (FN + TN + e)
    return {
        'Accuracy (%)':    accuracy    * 100,
        'Precision (%)':   precision   * 100,
        'Recall (%)':      recall      * 100,
        'Sensitivity (%)': recall      * 100,
        'Specificity (%)': specificity * 100,
        'F1':              f1,
        'FDR (%)':         fdr         * 100,
        'FPR/Fallout (%)': fpr         * 100,
        'FNR (%)':         fnr         * 100,
        'FOR (%)':         for_        * 100,
        'NPV (%)':         npv         * 100,
    }


## 9. CLM (Correlation Learning Mechanism)

In [ ]:
class CLM:
    """Sistem CLM lengkap: korelasi CNN <-> ANN.
    Catatan implementasi penting:
      - class_weights dihitung dari distribusi label training dan diteruskan ke
        setiap adam_step agar model tidak collapse ke kelas mayoritas.
      - metrik validasi (sensitivity/specificity/fallout) carry-forward nilai
        terakhir saat iterasi tidak dievaluasi, agar grafik tidak berosilasi ke 0.
      - best_ann diperbarui juga saat accuracy sama tetapi loss lebih kecil.
    """
    def __init__(self,
                 ann_layers:          Optional[List[int]] = None,
                 n_threads:           Optional[int]       = None,
                 n_palettes_per_iter: int   = 10,
                 iters_per_palette:   int   = 100,
                 total_iterations:    int   = 120_000,
                 adam_lr:             float = 0.00025,
                 adam_beta1:          float = 0.89,
                 adam_beta2:          float = 0.995,
                 adam_eps:            float = 1e-7,
                 bpta_lr:             float = 0.001,
                 dropout_p:           float = 0.10,
                 val_eval_freq:       int   = 5):
        self.ann_layers          = ann_layers or ANN.DEFAULT_LAYERS
        self.n_threads           = n_threads
        self.n_palettes_per_iter = n_palettes_per_iter
        self.iters_per_palette   = iters_per_palette
        self.total_iterations    = total_iterations
        self.adam_lr, self.adam_beta1, self.adam_beta2 = adam_lr, adam_beta1, adam_beta2
        self.adam_eps, self.bpta_lr, self.dropout_p    = adam_eps, bpta_lr, dropout_p
        self.val_eval_freq       = val_eval_freq

        self.cnn       = CNN()
        self.archives  = PaletteArchives()
        self.generator = PaletteGenerator(self.archives)

        self.best_palette:   Optional[Palette]      = None
        self.best_ann:       Optional[ANN]          = None
        self.best_accuracy:  float                  = 0.0
        self.best_loss:      float                  = float("inf")
        self._input_size:    Optional[int]          = None
        self._seg_cache:     Optional[torch.Tensor] = None
        self._val_seg_cache: Optional[torch.Tensor] = None
        self._class_weights: Optional[cp.ndarray]   = None

        self.history: Dict[str, List[float]] = {
            "iter": [], "accuracy": [], "loss": [],
            "sensitivity": [], "specificity": [], "fallout": [],
        }

    @staticmethod
    def _one_hot(labels: np.ndarray, nc: int = 2) -> np.ndarray:
        oh = np.zeros((len(labels), nc), dtype=np.float32)
        oh[np.arange(len(labels)), labels.astype(int)] = 1.0
        return oh

    @staticmethod
    def _compute_class_weights(labels: np.ndarray) -> cp.ndarray:
        # balanced class weight: w_c = n_total / (n_classes * n_c)
        # kelas minoritas mendapat bobot lebih besar agar model tidak bias
        n_total   = len(labels)
        n_classes = int(labels.max()) + 1
        weights   = np.zeros(n_classes, dtype=np.float32)
        for c in range(n_classes):
            n_c = int(np.sum(labels == c))
            weights[c] = n_total / (n_classes * max(n_c, 1))
        return cp.array(weights, dtype=cp.float32)

    def _extract(self, images: List[np.ndarray], palette: 'Palette',
                 desc: str = 'Extract',
                 seg_cache: Optional[torch.Tensor] = None) -> np.ndarray:
        if seg_cache is not None:
            return self.cnn.forward_batch(len(images), palette, seg_cache)
        return np.array(
            [self.cnn.forward(img, palette)
             for img in tqdm(images, desc=desc, leave=False, unit='img')],
            dtype=np.float32
        )

    def _fit_size(self, X: np.ndarray, target: int) -> np.ndarray:
        n, d = X.shape
        if d == target:  return X
        if d > target:   return X[:, :target]
        return np.hstack([X, np.zeros((n, target - d), dtype=np.float32)])

    def fit(self, train_images: List[np.ndarray], train_labels: np.ndarray,
            val_images:  Optional[List[np.ndarray]] = None,
            val_labels:  Optional[np.ndarray]       = None,
            verbose:     bool                       = True) -> None:
        y_train = self._one_hot(train_labels)
        self._class_weights = self._compute_class_weights(train_labels)

        mt = GPUBatchTrainer(
            n_threads=self.n_threads, iters_per_round=self.iters_per_palette,
            lr=self.adam_lr, beta1=self.adam_beta1, beta2=self.adam_beta2, eps=self.adam_eps,
        )
        budget      = self.total_iterations
        per_palette = self.iters_per_palette
        per_iter    = self.n_palettes_per_iter * per_palette
        n_outer     = max(10, budget // per_iter)

        # segmentasi dilakukan sekali di awal lalu di-cache di GPU
        self._seg_cache = self.cnn.pre_segment_images(train_images)
        if val_images is not None:
            self._val_seg_cache = self.cnn.pre_segment_images(val_images)

        # carry-forward metrik val terakhir saat iterasi tidak dievaluasi
        last_vm: Dict[str, float] = {}

        pbar_outer = tqdm(range(n_outer), desc='CLM Iterations', unit='iter')
        for it in pbar_outer:
            eff        = self.best_accuracy / 100.0
            candidates = self.generator.generate(self.n_palettes_per_iter, eff)
            best_it_acc, best_it_loss = -1.0, float("inf")
            best_it_pal, best_it_ann  = None, None

            for pal in tqdm(candidates, desc=f'Palettes [iter {it+1}]',
                            leave=False, unit='pal'):
                try:
                    X = self._extract(train_images, pal, seg_cache=self._seg_cache)
                except Exception as exc:
                    log.warning(f"Feature extraction error: {exc}")
                    continue
                if self._input_size is None:
                    self._input_size = X.shape[1]
                X = self._fit_size(X, self._input_size)
                seed_ann = (self.best_ann.clone() if self.best_ann is not None
                            else ANN(self._input_size, self.ann_layers, self.dropout_p))
                trained, loss, acc = mt.train_round(
                    seed_ann, X, y_train,
                    class_weights=self._class_weights,
                )
                if acc > best_it_acc or (acc == best_it_acc and loss < best_it_loss):
                    best_it_acc, best_it_loss = acc, loss
                    best_it_pal, best_it_ann  = pal, trained

            if best_it_pal is None:
                continue

            self.archives.add(best_it_acc, best_it_loss, best_it_pal)

            # update best jika accuracy naik, atau sama tapi loss lebih kecil
            if (best_it_acc * 100 > self.best_accuracy or
                    (abs(best_it_acc * 100 - self.best_accuracy) < 1e-6
                     and best_it_loss < self.best_loss)):
                self.best_accuracy = best_it_acc * 100
                self.best_loss     = best_it_loss
                self.best_palette  = best_it_pal
                self.best_ann      = best_it_ann

            vm: Dict[str, float] = {}
            if (val_images is not None and self.best_ann is not None
                    and it % self.val_eval_freq == 0):
                try:
                    Xv = self._extract(val_images, self.best_palette,
                                       seg_cache=self._val_seg_cache)
                    Xv = self._fit_size(Xv, self._input_size)
                    yv = self._one_hot(val_labels)
                    vm = compute_metrics(yv, cp.asnumpy(
                             self.best_ann.predict(cp.asarray(Xv))))
                    last_vm = vm
                except Exception:
                    pass

            effective_vm = vm if vm else last_vm

            self.history["iter"].append(it)
            self.history["accuracy"].append(best_it_acc * 100)
            self.history["loss"].append(best_it_loss)
            self.history["sensitivity"].append(effective_vm.get("Sensitivity (%)", 0))
            self.history["specificity"].append(effective_vm.get("Specificity (%)", 0))
            self.history["fallout"].append(effective_vm.get("FPR/Fallout (%)",    0))

            pbar_outer.set_postfix({
                'train_acc': f'{best_it_acc*100:.1f}%',
                'loss'     : f'{best_it_loss:.4f}',
                'best_acc' : f'{self.best_accuracy:.1f}%',
                'val_acc'  : (f'{effective_vm.get("Accuracy (%)", 0):.1f}%'
                               if effective_vm else 'N/A'),
            })

        print(f"Training selesai. Best accuracy: {self.best_accuracy:.2f}%"
              + (f" | palette: {self.best_palette.encode()}" if self.best_palette else ""))

    def predict(self, images: List[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
        """Return (probabilitas kelas, label biner). 0=sehat, 1=tumor."""
        if self.best_ann is None or self.best_palette is None:
            raise RuntimeError("Panggil .fit() sebelum .predict().")
        X    = self._extract(images, self.best_palette)
        X    = self._fit_size(X, self._input_size)
        prob = cp.asnumpy(self.best_ann.predict(cp.asarray(X)))
        return prob, np.argmax(prob, axis=1)

    def evaluate(self, images: List[np.ndarray], labels: np.ndarray) -> Dict[str, float]:
        prob, _ = self.predict(images)
        return compute_metrics(self._one_hot(labels), prob)

    def mark_detection(self, image: np.ndarray) -> Tuple[bool, np.ndarray]:
        """Return (tumor_detected, annotated_image)."""
        _, pred = self.predict([image])
        detected = bool(pred[0] == 1)
        ann_img  = image.copy()
        if detected:
            if ann_img.ndim == 2:
                ann_img = np.stack([ann_img] * 3, axis=-1)
            ann_img = ann_img.astype(np.float32)
            h, w = ann_img.shape[:2]
            t = max(2, min(h, w) // 50)
            ann_img[:t,  :,  0] = 1.0;  ann_img[:t,  :,  1:] = 0.0
            ann_img[-t:, :,  0] = 1.0;  ann_img[-t:, :,  1:] = 0.0
            ann_img[:,  :t,  0] = 1.0;  ann_img[:,  :t,  1:] = 0.0
            ann_img[:, -t:,  0] = 1.0;  ann_img[:, -t:,  1:] = 0.0
        return detected, ann_img


## 10. Data Augmentation


In [ ]:
# fungsi primitif transformasi (Iftikhar et al., 2025)
# input/output: grayscale float32 2D [0,1]; cv2 dipakai untuk BORDER_REFLECT_101 (tanpa artefak hitam)

def _to_uint8_gray(img: np.ndarray) -> np.ndarray:
    return np.clip(img * 255.0, 0, 255).astype(np.uint8)

def _to_float32_gray(img: np.ndarray) -> np.ndarray:
    return img.astype(np.float32) / 255.0


def apply_shearing(
    img        : np.ndarray,
    shear_range: tuple = (-15, 15),
    rng        : random.Random = None,
) -> np.ndarray:
    """Metode 1: shearing +-15 derajat sumbu x & y, pusat di tengah citra."""
    if rng is None:
        rng = random.Random()
    u8 = _to_uint8_gray(img)
    shear_x_deg = rng.uniform(shear_range[0], shear_range[1])
    shear_y_deg = rng.uniform(shear_range[0], shear_range[1])
    sx = np.tan(np.deg2rad(shear_x_deg))
    sy = np.tan(np.deg2rad(shear_y_deg))
    h, w = u8.shape
    cx, cy = w / 2.0, h / 2.0
    M = np.float32([[1, sx, -cy * sx],
                    [sy,  1, -cx * sy]])
    out = cv2.warpAffine(u8, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
    return _to_float32_gray(out)


def apply_rotation(
    img         : np.ndarray,
    angle_range : tuple = (-25, 25),
    rng         : random.Random = None,
) -> np.ndarray:
    """Metode 3: rotasi +-25 derajat berpusat di tengah citra."""
    if rng is None:
        rng = random.Random()
    u8 = _to_uint8_gray(img)
    angle = rng.uniform(angle_range[0], angle_range[1])
    h, w  = u8.shape
    cx, cy = w / 2.0, h / 2.0
    M = cv2.getRotationMatrix2D((cx, cy), angle, scale=1.0)
    out = cv2.warpAffine(u8, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
    return _to_float32_gray(out)


def apply_translation(
    img      : np.ndarray,
    tx_range : tuple = (-15, 15),
    ty_range : tuple = (-15, 15),
    rng      : random.Random = None,
) -> np.ndarray:
    """Translasi +-15 px sumbu x & y (komponen pipeline Metode 10)."""
    if rng is None:
        rng = random.Random()
    u8 = _to_uint8_gray(img)
    tx = rng.uniform(tx_range[0], tx_range[1])
    ty = rng.uniform(ty_range[0], ty_range[1])
    h, w = u8.shape
    M = np.float32([[1, 0, tx],
                    [0, 1, ty]])
    out = cv2.warpAffine(u8, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
    return _to_float32_gray(out)


def apply_translation_shearing_rotation(
    img         : np.ndarray,
    tx_range    : tuple = (-15, 15),
    ty_range    : tuple = (-15, 15),
    shear_range : tuple = (-15, 15),
    angle_range : tuple = (-25, 25),
    rng         : random.Random = None,
) -> np.ndarray:
    """Metode 10: Translasi -> Shearing -> Rotasi (pipeline paling kuat)."""
    img = apply_translation(img, tx_range, ty_range, rng)
    img = apply_shearing(img, shear_range, rng)
    img = apply_rotation(img, angle_range, rng)
    return img


# registry metode augmentasi: (kode_suffix, fungsi, kwargs_tambahan)
AUGMENTATION_METHODS_CLM = [
    ('shear',           apply_shearing,                      {}),
    ('rot',             apply_rotation,                      {}),
    ('trans_shear_rot', apply_translation_shearing_rotation, {}),
]


def augment_clm_dataset(
    images        : List[np.ndarray],
    labels        : np.ndarray,
    aug_per_method: int = 3,
    seed          : int = 42,
) -> Tuple[List[np.ndarray], np.ndarray]:
    """Augmentasi in-memory: aug_per_method salinan/gambar didistribusikan
    round-robin ke 3 metode. Output = len(images) x (1 + aug_per_method)."""
    rng       = random.Random(seed)
    n_methods = len(AUGMENTATION_METHODS_CLM)

    aug_imgs: List[np.ndarray] = list(images)
    aug_lbls: List[int]        = list(labels)

    for img, lbl in tqdm(zip(images, labels), total=len(images),
                         desc='Augmenting', unit='img'):
        for slot_idx in range(aug_per_method):
            _, aug_fn, kwargs = AUGMENTATION_METHODS_CLM[slot_idx % n_methods]
            aug_imgs.append(aug_fn(img, rng=rng, **kwargs))
            aug_lbls.append(int(lbl))

    return aug_imgs, np.array(aug_lbls, dtype=np.int32)


## 11. Dataset Loading


In [ ]:
SUPPORTED_IMAGE_EXTS = ('.npy', '.jpg', '.jpeg', '.png')

TARGET_IMG_SIZE = 128  # kelipatan CNN.GRID(=8) -> seg_sz=16, tanpa sisa piksel

def _resize_to_square(arr: np.ndarray, size: int) -> np.ndarray:
    """Resize grayscale float32 2D -> (size, size). Wajib agar seg_sz konsisten
    antar gambar (dipakai pre_segment_images()). Padding ke persegi dulu jika
    belum persegi (anchor kiri-atas) agar aspect ratio tidak terdistorsi."""
    h, w = arr.shape[:2]

    if h != w:
        pad_value = arr[0, 0]
        side = max(h, w)
        padded = np.full((side, side), pad_value, dtype=arr.dtype)
        padded[:h, :w] = arr
        arr = padded
        h = w = side

    interp = cv2.INTER_AREA if max(h, w) > size else cv2.INTER_CUBIC
    out = cv2.resize(arr, (size, size), interpolation=interp).astype(np.float32)
    return np.clip(out, 0.0, 1.0)  # clip karena INTER_CUBIC bisa overshoot


def _load_images_from_dir(directory: str) -> List[np.ndarray]:
    """Load semua gambar dari direktori sebagai float32 grayscale array."""
    paths: List[str] = []
    for ext in SUPPORTED_IMAGE_EXTS:
        paths.extend(glob.glob(os.path.join(directory, f'*{ext}')))
        paths.extend(glob.glob(os.path.join(directory, f'*{ext.upper()}')))
    paths = sorted(set(paths))

    if not paths:
        raise FileNotFoundError(
            f"Tidak ada file gambar ({', '.join(SUPPORTED_IMAGE_EXTS)}) "
            f"di '{directory}'."
        )

    imgs: List[np.ndarray] = []
    for p in tqdm(paths, desc=f'Loading {os.path.basename(directory)}', unit='file'):
        ext = os.path.splitext(p)[1].lower()
        try:
            if ext == '.npy':
                arr = np.load(p).astype(np.float32)
                if arr.ndim == 3:
                    arr = arr.mean(axis=-1)
                elif arr.ndim != 2:
                    log.warning(f"Skip {p}: dimensi tidak didukung ({arr.ndim}D)")
                    continue
            else:
                pil_img = Image.open(p).convert('L')
                arr = np.array(pil_img, dtype=np.float32)

            if arr.max() > 1.0:
                arr = arr / 255.0

            arr = _resize_to_square(arr, TARGET_IMG_SIZE)
            imgs.append(arr)

        except Exception as exc:
            log.warning(f"Gagal memuat '{p}': {exc} — dilewati")

    return imgs


_load_npy_from_dir = _load_images_from_dir  # alias backward-compatible


def load_dataset(
    train_no_tumor_dir: str = train_no_tumor,
    train_tumor_dir:    str = train_tumor,
    val_no_tumor_dir:   str = val_no_tumor,
    val_tumor_dir:      str = val_tumor,
    test_no_tumor_dir:  str = test_no_tumor,
    test_tumor_dir:     str = test_tumor,
) -> Tuple[
    Tuple[List[np.ndarray], np.ndarray],
    Tuple[List[np.ndarray], np.ndarray],
    Tuple[List[np.ndarray], np.ndarray],
]:
    """Load dataset train/val/test dari Google Drive. no_tumor=label 0, tumor=label 1."""

    def _load_split(no_t_dir, t_dir, split_name):
        no_t = _load_images_from_dir(no_t_dir)
        t    = _load_images_from_dir(t_dir)
        imgs = no_t + t
        lbls = np.array([0] * len(no_t) + [1] * len(t), dtype=np.int32)
        return imgs, lbls

    tr_imgs, tr_lbls   = _load_split(train_no_tumor_dir, train_tumor_dir,  'train')
    val_imgs, val_lbls = _load_split(val_no_tumor_dir,   val_tumor_dir,    'val')
    te_imgs, te_lbls   = _load_split(test_no_tumor_dir,  test_tumor_dir,   'test')

    print(f"Dataset: train={len(tr_imgs)}  val={len(val_imgs)}  test={len(te_imgs)}")
    return (tr_imgs, tr_lbls), (val_imgs, val_lbls), (te_imgs, te_lbls)


## 12. Load & Augment Dataset


In [ ]:
np.random.seed(AUG_SEED)
cp.random.seed(AUG_SEED)

(tr_imgs, tr_lbls), (val_imgs, val_lbls), (te_imgs, te_lbls) = load_dataset()

tr_imgs_aug, tr_lbls_aug = augment_clm_dataset(
    tr_imgs,
    tr_lbls,
    aug_per_method = AUG_PER_METHOD,
    seed           = AUG_SEED,
)

# shuffle dataset augmented
shuf_idx    = np.random.permutation(len(tr_imgs_aug))
tr_imgs_aug = [tr_imgs_aug[i] for i in shuf_idx]
tr_lbls_aug = tr_lbls_aug[shuf_idx]

# tr_imgs_orig/tr_lbls_orig = data asli (untuk evaluasi per-fold CV)
# tr_imgs/tr_lbls           = data augmented (untuk training CLM)
tr_imgs_orig = tr_imgs
tr_lbls_orig = tr_lbls
tr_imgs      = tr_imgs_aug
tr_lbls      = tr_lbls_aug

print(f"Train (augmented): {len(tr_imgs)} | Val: {len(val_imgs)} | Test: {len(te_imgs)}")


## 13. Initialise & Train CLM + Grid Search Hyperparameter


In [ ]:
import itertools

if REDUCED_GRID:
    combinations_clm = REDUCED_COMBINATIONS_CLM
else:
    combinations_clm = list(itertools.product(
        PARAM_GRID_CLM['adam_lr'],
        PARAM_GRID_CLM['dropout_p'],
        PARAM_GRID_CLM['total_iter'],
    ))

total_runs = len(combinations_clm)

import pandas as pd
df_grid_clm = pd.DataFrame(
    combinations_clm,
    columns=['adam_lr', 'dropout_p', 'total_iter']
)
df_grid_clm.index = df_grid_clm.index + 1
df_grid_clm.index.name = 'Run'
print(f'Total kombinasi: {total_runs}')
print(df_grid_clm.to_string())


In [ ]:
import json as _json_gs
import time as _time_gs

GS_RESULTS_JSON = f'{OUTPUT_DIR}/clm_grid_results.json'

# resume dari hasil sebelumnya jika ada
if os.path.exists(GS_RESULTS_JSON):
    with open(GS_RESULTS_JSON, 'r') as _f:
        gs_results = _json_gs.load(_f)
    done_names_gs = {r['name'] for r in gs_results}
else:
    gs_results    = []
    done_names_gs = set()

def _gs_combo_name(lr, dp, ti):
    return f'lr{lr}_dp{dp}_ti{ti}'

gs_start = _time_gs.time()

for run_idx, (lr, dp, ti) in enumerate(combinations_clm, start=1):
    name = _gs_combo_name(lr, dp, ti)

    if name in done_names_gs:
        continue

    run_start = _time_gs.time()

    clm_gs = CLM(
        ann_layers          = ANN.DEFAULT_LAYERS,
        n_threads           = None,
        n_palettes_per_iter = 4,
        iters_per_palette   = 30,
        total_iterations    = ti,
        adam_lr             = lr,
        adam_beta1          = 0.89,
        adam_beta2          = 0.995,
        adam_eps            = 1e-7,
        bpta_lr             = 0.001,
        dropout_p           = dp,
        val_eval_freq       = 5,
    )

    clm_gs.fit(
        tr_imgs, tr_lbls,
        val_images = val_imgs,
        val_labels = val_lbls,
        verbose    = False,
    )

    test_metrics = clm_gs.evaluate(te_imgs, te_lbls)
    val_metrics  = clm_gs.evaluate(val_imgs, val_lbls)

    run_time = _time_gs.time() - run_start

    result = {
        'run'           : run_idx,
        'name'          : name,
        'adam_lr'       : lr,
        'dropout_p'     : dp,
        'total_iter'    : ti,
        'best_accuracy' : round(clm_gs.best_accuracy, 4),
        'best_loss'     : round(clm_gs.best_loss,     6),
        'val_acc'       : round(val_metrics.get('Accuracy (%)', 0.0),     4),
        'val_sens'      : round(val_metrics.get('Sensitivity (%)', 0.0),  4),
        'val_spec'      : round(val_metrics.get('Specificity (%)', 0.0),  4),
        'test_acc'      : round(test_metrics.get('Accuracy (%)', 0.0),    4),
        'test_sens'     : round(test_metrics.get('Sensitivity (%)', 0.0), 4),
        'test_spec'     : round(test_metrics.get('Specificity (%)', 0.0), 4),
        'test_f1'       : round(test_metrics.get('F1', 0.0),               4),
        'test_precision': round(test_metrics.get('Precision (%)', 0.0),  4),
        'best_palette'  : clm_gs.best_palette.encode() if clm_gs.best_palette else 'N/A',
        'run_time_sec'  : round(run_time, 1),
        'history_train_acc' : clm_gs.history['accuracy'],
        'history_loss'      : clm_gs.history['loss'],
        'history_sens'      : clm_gs.history['sensitivity'],
        'history_spec'      : clm_gs.history['specificity'],
    }

    gs_results.append(result)
    done_names_gs.add(name)

    # auto-save tiap run agar tahan crash
    with open(GS_RESULTS_JSON, 'w') as _f:
        _json_gs.dump(gs_results, _f, indent=2)

    print(f'[{run_idx:02d}/{total_runs}] {name}  '
          f'val_acc={result["val_acc"]:.2f}%  test_acc={result["test_acc"]:.2f}%  '
          f'test_f1={result["test_f1"]:.2f}%  ({run_time/60:.1f} mnt)')

total_gs_time = (_time_gs.time() - gs_start) / 60
print(f'Grid Search selesai. Total waktu: {total_gs_time:.1f} menit. Hasil: {GS_RESULTS_JSON}')


## 13b. Tabel Ringkasan Hasil Grid Search


In [ ]:
import json as _json_sum
import pandas as pd
import matplotlib.pyplot as plt

with open(GS_RESULTS_JSON, 'r') as _f:
    gs_results_loaded = _json_sum.load(_f)

cols_gs = ['run', 'adam_lr', 'dropout_p', 'total_iter',
           'val_acc', 'test_acc', 'test_precision', 'test_f1',
           'test_sens', 'test_spec', 'best_loss', 'run_time_sec']

df_gs = pd.DataFrame([{c: r.get(c, 0.0) for c in cols_gs} for r in gs_results_loaded])

# --- Composite score: rata-rata 5 metrik evaluasi yang diminta ---
# Accuracy, Precision, Recall (=Sensitivity), Specificity, F1
# Semua disamakan ke skala 0-100 (compute_metrics 'F1' masih 0-1, x100)
df_gs['composite_score'] = (
    df_gs['test_acc']
    + df_gs['test_precision']
    + df_gs['test_sens']
    + df_gs['test_spec']
    + (df_gs['test_f1'] * 100)
) / 5.0

# Model TERBAIK = ranking berdasarkan composite_score (5 metrik), bukan test_acc saja
df_gs = df_gs.sort_values('composite_score', ascending=False).reset_index(drop=True)

# Format untuk display
df_gs['test_acc_%']   = df_gs['test_acc'].round(2)
df_gs['val_acc_%']    = df_gs['val_acc'].round(2)
df_gs['test_prec_%']  = df_gs['test_precision'].round(2)
df_gs['test_f1_%']    = (df_gs['test_f1'] * 100).round(2)
df_gs['test_sens_%']  = df_gs['test_sens'].round(2)
df_gs['test_spec_%']  = df_gs['test_spec'].round(2)
df_gs['composite_%']  = df_gs['composite_score'].round(2)
df_gs['time_min']     = (df_gs['run_time_sec'] / 60).round(1)

display_cols_gs = ['run', 'adam_lr', 'dropout_p', 'total_iter',
                   'val_acc_%', 'test_acc_%', 'test_prec_%', 'test_f1_%',
                   'test_sens_%', 'test_spec_%', 'composite_%', 'time_min']

print('Hasil Grid Search CLM (diurutkan berdasarkan Composite Score):')
try:
    display(df_gs[display_cols_gs].rename(columns={
        'adam_lr'     : 'LR',
        'dropout_p'   : 'Dropout',
        'total_iter'  : 'Iter',
        'val_acc_%'   : 'Val Acc (%)',
        'test_acc_%'  : 'Accuracy (%)',
        'test_prec_%' : 'Precision (%)',
        'test_f1_%'   : 'F1 (%)',
        'test_sens_%' : 'Recall (%)',
        'test_spec_%' : 'Specificity (%)',
        'composite_%' : 'Composite (%)',
        'time_min'    : 'Waktu (mnt)',
    }))
except NameError:
    print(df_gs[display_cols_gs].to_string())

# Kombinasi terbaik (berdasarkan Composite Score: rata-rata
# Accuracy, Precision, Recall, Specificity, F1)
best_gs = df_gs.iloc[0]
print(f'\nKombinasi terbaik: LR={best_gs.adam_lr}  Dropout={best_gs.dropout_p}  '
      f'Iter={best_gs.total_iter}  |  Accuracy={best_gs["test_acc_%"]:.2f}%  '
      f'Composite={best_gs["composite_%"]:.2f}%')

csv_gs_path = f'{OUTPUT_DIR}/clm_grid_results_summary.csv'
df_gs[display_cols_gs].to_csv(csv_gs_path, index=False)
print(f'Tabel disimpan ke: {csv_gs_path}')


## 13c. Visualisasi Grid Search


In [ ]:
import matplotlib.pyplot as plt

# plot 1: test accuracy per kombinasi
fig, ax = plt.subplots(figsize=(max(10, len(df_gs) * 0.9), 5))

labels_gs = [
    f"lr={r['adam_lr']}\ndp={r['dropout_p']}\nit={r['total_iter']}"
    for _, r in df_gs.iterrows()
]
colors_gs = ['#2ecc71' if i == 0 else '#3498db' if i < 3 else '#95a5a6'
             for i in range(len(df_gs))]

bars = ax.bar(range(len(df_gs)), df_gs['test_acc_%'],
              color=colors_gs, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, df_gs['test_acc_%']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(range(len(df_gs)))
ax.set_xticklabels(labels_gs, fontsize=7)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('CLM Grid Search -- Test Accuracy per Kombinasi Hyperparameter\n'
             'Hijau = Terbaik  |  Biru = Top-3', fontweight='bold')
ax.set_ylim(max(0, df_gs['test_acc_%'].min() - 5),
            min(101, df_gs['test_acc_%'].max() + 3))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/clm_grid_bar_test_acc.png', dpi=150)
plt.show()

# plot 2: pengaruh tiap hyperparameter
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
fig2.suptitle('Pengaruh Tiap Hyperparameter CLM terhadap Test Accuracy',
              fontsize=13, fontweight='bold')

for param, label, ax in [
    ('adam_lr',    'Learning Rate', axes2[0]),
    ('dropout_p',  'Dropout Rate',  axes2[1]),
    ('total_iter', 'Total Iter',    axes2[2]),
]:
    grouped = df_gs.groupby(param)['test_acc_%']
    means   = grouped.mean()
    stds    = grouped.std().fillna(0)
    ax.bar(range(len(means)), means.values,
           yerr=stds.values, capsize=4,
           color='#3498db', edgecolor='white')
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels([str(v) for v in means.index], fontsize=9)
    ax.set_xlabel(label)
    ax.set_ylabel('Rata-rata Test Accuracy (%)')
    ax.grid(axis='y', alpha=0.3)
    # Anotasi nilai rata-rata
    for i, (v, s) in enumerate(zip(means.values, stds.values)):
        ax.text(i, v + s + 0.3, f'{v:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/clm_grid_param_effect.png', dpi=150)
plt.show()
print('Plot disimpan.')


## 13d. Training Ulang dengan Kombinasi Terbaik


In [ ]:
import time

best_gs_row = df_gs.iloc[0]
BEST_LR     = float(best_gs_row['adam_lr'])
BEST_DP     = float(best_gs_row['dropout_p'])
BEST_TI     = int(best_gs_row['total_iter'])

clm = CLM(
    ann_layers          = ANN.DEFAULT_LAYERS,
    n_threads           = None,
    n_palettes_per_iter = 4,
    iters_per_palette   = 30,
    total_iterations    = BEST_TI,
    adam_lr             = BEST_LR,
    adam_beta1          = 0.89,
    adam_beta2          = 0.995,
    adam_eps            = 1e-7,
    bpta_lr             = 0.001,
    dropout_p           = BEST_DP,
    val_eval_freq       = 5,
)

t0 = time.time()
clm.fit(
    tr_imgs, tr_lbls,
    val_images = val_imgs,
    val_labels = val_lbls,
    verbose    = True,
)
elapsed = time.time() - t0

print(f'Hyperparameter: LR={BEST_LR}  Dropout={BEST_DP}  Iter={BEST_TI}')
print(f'Training time: {elapsed/60:.2f} menit')


## 13e. Select-Shuffle-Test CV


In [ ]:
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

# parameter Select-Shuffle-Test CV
K_SST            = 5      # jumlah fold
SST_ITERATIONS   = 3      # jumlah ulangan shuffle-test
SST_SEED_BASE    = 100
SST_AUG_MULTI    = TARGET_MULTIPLIER

# gabungkan semua data asli (sebelum augmentasi): train_orig + val + test
all_imgs_orig  = tr_imgs_orig + val_imgs + te_imgs
all_lbls_orig  = np.concatenate([tr_lbls_orig, val_lbls, te_lbls])


def augment_fold_train(imgs, lbls, aug_multiplier, seed):
    """Augmentasi hanya pada fold training (dipanggil setelah split fold,
    mencegah data leakage ke fold val/test)."""
    aug_per_method = aug_multiplier - 1
    if aug_per_method <= 0:
        return imgs, lbls
    imgs_aug, lbls_aug = augment_clm_dataset(
        imgs, lbls,
        aug_per_method=aug_per_method,
        seed=seed,
    )
    shuf = np.random.permutation(len(imgs_aug))
    return [imgs_aug[i] for i in shuf], lbls_aug[shuf]


def run_clm_fold(tr_i, tr_l, te_i, te_l, fold_idx, iter_idx):
    """Training satu fold CLM dengan HP terbaik, evaluasi di fold test."""
    clm_fold = CLM(
        ann_layers          = ANN.DEFAULT_LAYERS,
        n_threads           = None,
        n_palettes_per_iter = 4,
        iters_per_palette   = 30,
        total_iterations    = BEST_TI,
        adam_lr             = BEST_LR,
        adam_beta1          = 0.89,
        adam_beta2          = 0.995,
        adam_eps            = 1e-7,
        bpta_lr             = 0.001,
        dropout_p           = BEST_DP,
        val_eval_freq       = 5,
    )
    clm_fold.fit(tr_i, tr_l, verbose=False)
    metrics = clm_fold.evaluate(te_i, te_l)
    metrics['fold']      = fold_idx
    metrics['iteration'] = iter_idx
    del clm_fold
    if USE_GPU:
        cp.get_default_memory_pool().free_all_blocks()
    return metrics


# FASE 1 (SELECT): k-fold pertama untuk konfirmasi HP terbaik
skf_select = StratifiedKFold(n_splits=K_SST, shuffle=True, random_state=SST_SEED_BASE)
select_results = []
indices = np.arange(len(all_imgs_orig))

for fold_idx, (train_idx, test_idx) in enumerate(skf_select.split(indices, all_lbls_orig)):
    fold_tr_imgs = [all_imgs_orig[i] for i in train_idx]
    fold_tr_lbls = all_lbls_orig[train_idx]
    fold_te_imgs = [all_imgs_orig[i] for i in test_idx]
    fold_te_lbls = all_lbls_orig[test_idx]

    # augmentasi hanya pada training fold agar tidak leakage ke test fold
    aug_seed = SST_SEED_BASE + fold_idx
    fold_tr_imgs_aug, fold_tr_lbls_aug = augment_fold_train(
        fold_tr_imgs, fold_tr_lbls, SST_AUG_MULTI, aug_seed
    )

    metrics = run_clm_fold(
        fold_tr_imgs_aug, fold_tr_lbls_aug,
        fold_te_imgs,     fold_te_lbls,
        fold_idx=fold_idx+1, iter_idx=0
    )
    select_results.append(metrics)

print(f'SST CV — data asli: {len(all_imgs_orig)} gambar  |  K={K_SST}  |  '
      f'HP: lr={BEST_LR}, dropout={BEST_DP}, iter={BEST_TI}')
print(f'Fase SELECT selesai: {len(select_results)} fold.')


In [ ]:
# FASE 2 (SHUFFLE -> TEST): data diacak ulang SST_ITERATIONS kali, tiap kali di-k-fold lagi
sst_all_results = []
sst_iter_means  = []

sst_start = time.time()

for sst_iter in range(SST_ITERATIONS):
    iter_seed = SST_SEED_BASE + 1000 + sst_iter * 100

    rng = np.random.default_rng(iter_seed)
    shuffled_idx = rng.permutation(len(all_imgs_orig))
    imgs_shuffled = [all_imgs_orig[i] for i in shuffled_idx]
    lbls_shuffled = all_lbls_orig[shuffled_idx]

    skf_test = StratifiedKFold(n_splits=K_SST, shuffle=True, random_state=iter_seed)
    indices_sh = np.arange(len(imgs_shuffled))
    iter_fold_results = []

    for fold_idx, (train_idx, test_idx) in enumerate(
            skf_test.split(indices_sh, lbls_shuffled)):

        fold_tr_imgs = [imgs_shuffled[i] for i in train_idx]
        fold_tr_lbls = lbls_shuffled[train_idx]
        fold_te_imgs = [imgs_shuffled[i] for i in test_idx]
        fold_te_lbls = lbls_shuffled[test_idx]

        # augmentasi hanya pada training fold
        aug_seed = iter_seed + fold_idx
        fold_tr_imgs_aug, fold_tr_lbls_aug = augment_fold_train(
            fold_tr_imgs, fold_tr_lbls, SST_AUG_MULTI, aug_seed
        )

        metrics = run_clm_fold(
            fold_tr_imgs_aug, fold_tr_lbls_aug,
            fold_te_imgs,     fold_te_lbls,
            fold_idx=fold_idx+1, iter_idx=sst_iter+1
        )
        iter_fold_results.append(metrics)
        sst_all_results.append(metrics)

    iter_mean = {
        'iteration'       : sst_iter + 1,
        'Accuracy (%)'    : np.mean([r['Accuracy (%)']    for r in iter_fold_results]),
        'Precision (%)'   : np.mean([r['Precision (%)']   for r in iter_fold_results]),
        'Recall (%)'      : np.mean([r['Recall (%)']      for r in iter_fold_results]),
        'Sensitivity (%)' : np.mean([r['Sensitivity (%)'] for r in iter_fold_results]),
        'Specificity (%)' : np.mean([r['Specificity (%)'] for r in iter_fold_results]),
        'F1-Score (%)'    : np.mean([r['F1-Score (%)']    for r in iter_fold_results]),
    }
    sst_iter_means.append(iter_mean)
    print(f'Iterasi {sst_iter+1}/{SST_ITERATIONS}: acc={iter_mean["Accuracy (%)"]:.2f}%  '
          f'F1={iter_mean["F1-Score (%)"]:.2f}%  sens={iter_mean["Sensitivity (%)"]:.2f}%')

total_sst_time = (time.time() - sst_start) / 60
print(f'Fase SHUFFLE-TEST selesai. Total waktu: {total_sst_time:.1f} menit')


In [ ]:
import json as _json_sst

# compute_metrics mengembalikan 'F1' (0-1); tambahkan versi persen agar konsisten
for res in sst_all_results:
    if 'F1' in res and 'F1-Score (%)' not in res:
        res['F1-Score (%)'] = res['F1'] * 100

metric_keys = ['Accuracy (%)', 'Precision (%)', 'Recall (%)',
               'Sensitivity (%)', 'Specificity (%)', 'F1-Score (%)']

print(f'Hasil Select-Shuffle-Test CV — {len(sst_all_results)} fold '
      f'({SST_ITERATIONS} iterasi x {K_SST} fold)')

sst_summary = {}
print(f'  {"Metrik":<22}  {"Mean":>8}  {"Std":>7}  {"95% CI":>20}')
for k in metric_keys:
    vals   = [r[k] for r in sst_all_results]
    mean_v = np.mean(vals)
    std_v  = np.std(vals)
    ci_lo  = mean_v - 1.96 * std_v / np.sqrt(len(vals))
    ci_hi  = mean_v + 1.96 * std_v / np.sqrt(len(vals))
    sst_summary[k] = {'mean': mean_v, 'std': std_v, 'ci_lo': ci_lo, 'ci_hi': ci_hi}
    print(f'  {k:<22}  {mean_v:>7.2f}%  {std_v:>6.2f}%  [{ci_lo:>6.2f}%, {ci_hi:>6.2f}%]')

# perbandingan vs model final one-time split (section 13d)
final_metrics_onetime = clm.evaluate(te_imgs, te_lbls)
if 'F1' in final_metrics_onetime and 'F1-Score (%)' not in final_metrics_onetime:
    final_metrics_onetime['F1-Score (%)'] = final_metrics_onetime['F1'] * 100

print(f'\n  {"Metrik":<22}  {"One-time split":>15}  {"SST CV (mean±std)":>22}')
for k in metric_keys:
    ot_val  = final_metrics_onetime.get(k, 0.0)
    cv_mean = sst_summary[k]['mean']
    cv_std  = sst_summary[k]['std']
    gap     = ot_val - cv_mean
    flag    = '  <- overoptimis' if gap > 3.0 else ''
    print(f'  {k:<22}  {ot_val:>14.2f}%  {cv_mean:>9.2f}% ± {cv_std:>5.2f}%{flag}')

df_sst = pd.DataFrame(sst_all_results)
df_sst_display = df_sst[['iteration', 'fold'] + metric_keys].round(2)
try:
    display(df_sst_display)
except NameError:
    print(df_sst_display.to_string())

sst_json_path = f'{OUTPUT_DIR}/clm_sst_cv_results.json'
with open(sst_json_path, 'w') as _f:
    _json_sst.dump({
        'parameters'   : {'K': K_SST, 'iterations': SST_ITERATIONS,
                          'best_lr': BEST_LR, 'best_dp': BEST_DP, 'best_ti': BEST_TI},
        'summary'      : {k: sst_summary[k] for k in metric_keys},
        'all_fold_results': [{
            'iteration': r['iteration'], 'fold': r['fold'],
            **{k: round(r[k], 4) for k in metric_keys}
        } for r in sst_all_results]
    }, _f, indent=2)

sst_csv_path = f'{OUTPUT_DIR}/clm_sst_cv_results.csv'
df_sst_display.to_csv(sst_csv_path, index=False)

print(f'\nDisimpan: {sst_json_path} | {sst_csv_path}')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    f'Select-Shuffle-Test CV — CLM (k={K_SST}, {SST_ITERATIONS} iterasi)\n'
    'Estimasi generalization performance dengan hyperparameter terbaik dari grid search',
    fontsize=12, fontweight='bold'
)

# plot 1: boxplot accuracy per iterasi
ax = axes[0]
iter_acc_data = [
    [r['Accuracy (%)'] for r in sst_all_results if r['iteration'] == i+1]
    for i in range(SST_ITERATIONS)
]
bp = ax.boxplot(iter_acc_data, patch_artist=True,
                boxprops=dict(facecolor='#dbeafe', color='#1e40af'),
                medianprops=dict(color='#1e40af', linewidth=2),
                whiskerprops=dict(color='#1e40af'),
                capprops=dict(color='#1e40af'))
ax.set_xticklabels([f'Iter {i+1}' for i in range(SST_ITERATIONS)])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy per iterasi shuffle')
ax.axhline(np.mean([r['Accuracy (%)'] for r in sst_all_results]),
           color='red', linestyle='--', linewidth=1, alpha=0.7, label='Grand mean')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# plot 2: bar + error bar metrik utama
ax2 = axes[1]
plot_keys  = ['Accuracy (%)', 'Sensitivity (%)', 'Specificity (%)', 'F1-Score (%)']
plot_means = [sst_summary[k]['mean'] for k in plot_keys]
plot_stds  = [sst_summary[k]['std']  for k in plot_keys]
colors_bar = ['#2563eb', '#16a34a', '#9333ea', '#ea580c']
bars = ax2.bar(range(len(plot_keys)), plot_means, yerr=plot_stds, capsize=5,
               color=colors_bar, alpha=0.8, edgecolor='white')
ax2.set_xticks(range(len(plot_keys)))
ax2.set_xticklabels(['Accuracy', 'Sensitivity', 'Specificity', 'F1'], fontsize=9)
ax2.set_ylabel('Score (%)')
ax2.set_title('Mean ± Std metrik utama (SST CV)')
for bar, mean_v, std_v in zip(bars, plot_means, plot_stds):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std_v + 0.3,
             f'{mean_v:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax2.set_ylim(0, min(110, max(plot_means) + max(plot_stds) + 8))
ax2.grid(axis='y', alpha=0.3)

# plot 3: perbandingan one-time split vs SST CV
ax3 = axes[2]
comp_keys  = ['Accuracy (%)', 'Sensitivity (%)', 'Specificity (%)', 'F1-Score (%)']
ot_vals    = [final_metrics_onetime.get(k, 0) for k in comp_keys]
cv_means   = [sst_summary[k]['mean'] for k in comp_keys]
cv_stds    = [sst_summary[k]['std']  for k in comp_keys]
x_pos      = np.arange(len(comp_keys))
width      = 0.35
ax3.bar(x_pos - width/2, ot_vals, width, label='One-time split',
        color='#f97316', alpha=0.8, edgecolor='white')
ax3.bar(x_pos + width/2, cv_means, width, yerr=cv_stds, capsize=4,
        label='SST CV (mean±std)', color='#2563eb', alpha=0.8, edgecolor='white')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(['Accuracy', 'Sensitivity', 'Specificity', 'F1'], fontsize=9)
ax3.set_ylabel('Score (%)')
ax3.set_title('One-time split vs SST CV')
ax3.legend(fontsize=9)
ax3.set_ylim(0, min(110, max(ot_vals + cv_means) + 10))
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plot_path = f'{OUTPUT_DIR}/clm_sst_cv_plot.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot disimpan: {plot_path}')


## 14. Evaluation

In [ ]:
if clm.best_ann:
    metrics = clm.evaluate(te_imgs, te_lbls)
    print(f"{'Metric':<22}  Value")
    for k, v in metrics.items():
        print(f"{k:<22}  {v:.2f}")
else:
    print("Model belum dilatih.")


## 16. Training History

In [ ]:
if clm.history["iter"]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(clm.history["iter"], clm.history["accuracy"], color="steelblue")
    axes[0].set_title("Training Accuracy (%)")
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Accuracy (%)")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(clm.history["iter"], clm.history["loss"], color="tomato")
    axes[1].set_title("Training Loss")
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Cross-Entropy Loss")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(clm.history["iter"], clm.history["sensitivity"],  label="Sensitivity", color="green")
    axes[2].plot(clm.history["iter"], clm.history["specificity"],  label="Specificity", color="purple")
    axes[2].plot(clm.history["iter"], clm.history["fallout"],      label="FPR/Fallout", color="orange")
    axes[2].set_title("Val Sensitivity / Specificity / FPR")
    axes[2].set_xlabel("Iteration")
    axes[2].set_ylabel("%")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.suptitle("CLM Training History", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"Best palette : {clm.best_palette.encode() if clm.best_palette else 'N/A'}")
    print(f"Best accuracy: {clm.best_accuracy:.2f}%")

## 17. Simpan Model & Hasil

In [ ]:
import pickle, json as _json

# simpan objek CLM (best_ann, palette, history) via pickle
clm_save_path = f'{OUTPUT_DIR}/clm_model.pkl'
with open(clm_save_path, 'wb') as _f:
    pickle.dump({
        'best_ann'      : clm.best_ann,
        'best_palette'  : clm.best_palette,
        'best_accuracy' : clm.best_accuracy,
        'history'       : clm.history,
    }, _f)

# simpan training history sebagai JSON
hist_path = f'{OUTPUT_DIR}/clm_history.json'
with open(hist_path, 'w') as _f:
    _json.dump(clm.history, _f, indent=2)

# simpan plot training history
if clm.history['iter']:
    import matplotlib.pyplot as _plt
    _fig, _axes = _plt.subplots(1, 3, figsize=(15, 4))
    _axes[0].plot(clm.history['iter'], clm.history['accuracy'], color='steelblue')
    _axes[0].set_title('Training Accuracy (%)')
    _axes[0].set_xlabel('Iteration'); _axes[0].set_ylabel('Accuracy (%)')
    _axes[0].grid(True, alpha=0.3)
    _axes[1].plot(clm.history['iter'], clm.history['loss'], color='tomato')
    _axes[1].set_title('Training Loss')
    _axes[1].set_xlabel('Iteration'); _axes[1].set_ylabel('Cross-Entropy Loss')
    _axes[1].grid(True, alpha=0.3)
    _axes[2].plot(clm.history['iter'], clm.history['sensitivity'],  label='Sensitivity', color='green')
    _axes[2].plot(clm.history['iter'], clm.history['specificity'],  label='Specificity', color='purple')
    _axes[2].plot(clm.history['iter'], clm.history['fallout'],      label='FPR/Fallout', color='orange')
    _axes[2].set_title('Val Sensitivity / Specificity / FPR')
    _axes[2].set_xlabel('Iteration'); _axes[2].set_ylabel('%'); _axes[2].legend()
    _axes[2].grid(True, alpha=0.3)
    _plt.suptitle('CLM Training History', fontsize=13, fontweight='bold')
    _plt.tight_layout()
    hist_fig_path = f'{OUTPUT_DIR}/clm_training_history.png'
    _plt.savefig(hist_fig_path, dpi=150)
    _plt.show()

print(f'Semua output tersimpan di: {OUTPUT_DIR}')
